# Lab 15: Deploy Your Model, a Missing-Item Alarm

**Duration:** 3 hours. **Level:** beginner. **Work mode:** pairs or individual.

| Student name | Station no. | Date |
|---|---|---|
| | | |

## 1. Overview
In Lab 14 you trained your own model. A model on its own is not yet a product. In this lab you **deploy** it: you make it fast with TensorRT (Lab 8), and you build an application around it. The application is a **tool board** or **shadow board**: a board with a place for each tool. When a tool is **missing** from the board for too long, the program raises an **alarm**, writes it in a log and can send it to your phone. Workshops, labs and clinics use this to stop tools from getting lost.

**By the end of this lab you can:**
1. Convert your own model to a TensorRT FP16 engine and measure the speed-up.
2. Check which required items are inside a zone and decide when an item is missing.
3. Use a hold time for each item so the alarm is steady.
4. Log every missing and returned event with the time away.
5. Send the alarm to a phone with MQTT (optional).

**How this notebook works.** Run a cell with **Shift + Enter**, from the top. Type your answers in the cells that say *Your answers*. The live video appears under the cell that starts it. Press **Ctrl + S** often.

> **Before you start.** You need the model from Lab 14 (`runs/mine/weights/best.pt`). If you do not have it, the notebook uses the general model `yolo26n.pt` and you choose items from the 80 classes of Lab 7 (for example cup, scissors, cell phone, bottle).

## 2. Background

**From model to product.** Four things make a model useful in the real world:

| Step | In this lab |
|---|---|
| Speed | The model must run fast enough: TensorRT FP16 engine (Part B) |
| Decision | The program turns detections into a decision: is each item on the board? (Part C) |
| Stability | The decision must not flicker: a hold time for each item (Part C) |
| Record and report | Every event is logged and can be sent to a phone (Parts C and D) |

**An engine for a custom model.** Lab 8 built an engine for `yolo26n`. You build one for your own model the same way. An engine is made for **one picture size** (`IMGSZ`, the size you trained with, for example 320) and for **one kind of Jetson**, so you build it on this Jetson. The engine file `mine.engine` gives the same results as `mine.pt` but faster.

**Is an item on the board?** For each detection the program takes the **centre of the box**. If the centre is inside the zone (the board) and the confidence is at least `CONFIDENCE`, the item is **on the board**. `cv2.pointPolygonTest` tells if a point is inside a polygon.

**The hold time, for every item.** Each required item has its own state, `OK` or `MISSING`. The state changes only after the new situation lasted `HOLD` seconds. If you pick up a tool for two seconds, nothing happens. If you take it away for longer, the alarm is raised. When the tool comes back, the program logs how long it was missing.

| | If you raise `HOLD` | If you lower `HOLD` |
|---|---|---|
| Effect | Fewer false alarms, but a late alarm | Fast alarms, but flicker when the model misses the item for a moment |

**MQTT (optional).** MQTT sends short messages between programs. Your Jetson has its own MQTT broker (see Lab 9's notes and the Details of your student card on the teacher page). A phone app such as *IoT MQTT Panel* can subscribe to a topic and show the alarm on the phone.

**Change one thing at a time.** If you break the notebook, close it and run `import labview; labview.reset(15)` in a cell of a new notebook (File, New, Notebook) for a fresh copy.

---
# Part A: Prepare and Plan (20 min)

### Step 1: Get Your Model
The cell looks for the model of Lab 14 and copies it here as `mine.pt`. If it is not there, the notebook uses `yolo26n.pt`.

In [ ]:
import csv
import glob
import json
import os
import shutil
import subprocess
import sys
import time
from datetime import datetime

import cv2
import numpy as np

import labview

print("This notebook is saved in:", os.getcwd())

for source in ["../lab14/runs/mine/weights/best.pt", "../lab14/mine.pt"]:
    if os.path.exists(source) and not os.path.exists("mine.pt"):
        shutil.copy(source, "mine.pt")
        print("copied", source)

if os.path.exists("mine.pt"):
    MODEL_PT, IMGSZ = "mine.pt", 320
    print("Your model from Lab 14 is here: mine.pt")
else:
    for source in ["../models/yolo26n.pt", "../lab8/yolo26n.pt", "../lab9/yolo26n.pt"]:
        if os.path.exists(source) and not os.path.exists("yolo26n.pt"):
            shutil.copy(source, "yolo26n.pt")
    MODEL_PT, IMGSZ = "yolo26n.pt", 640
    print("No model from Lab 14. Using yolo26n.pt: choose REQUIRED from the 80 classes of Lab 7.")
print("Model:", MODEL_PT, "| picture size:", IMGSZ)

### Step 2: Plan Your Board
Choose what the board is and which items must be on it.

| Idea | Board | Required items |
|---|---|---|
| Tool board | A sheet on the wall with a place for each tool | Your own objects from Lab 14 |
| Lab kit check | A tray | scissors, cup, cell phone (general model) |
| Desk check before you leave | The desk | bottle, cup, cell phone |

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Your board idea:
- Which items are required?:
- Who should get the alarm and what should they do?:


---
# Part B: Make the Model Fast (45 min)

### Step 1: Build the Engine
The cell writes a small program `export_mine.py` that builds the FP16 engine, like the one of Lab 8, for your model and picture size. Then it runs the program in the lock, so only one student builds at a time. The build takes **several minutes** and the output has many lines. Red or yellow `[TRT]` lines are normal.

> **Warning.** Do not press the stop button and do not unplug anything. Wait until you see `Built mine.engine`.

In [ ]:
%%writefile export_mine.py
import os
import sys
import time

from ultralytics import YOLO

pt, imgsz = sys.argv[1], int(sys.argv[2])
model = YOLO(pt)
start = time.time()
try:
    path = model.export(format="engine", device=0, imgsz=imgsz, quantize=16)
except Exception as error:                         # older Ultralytics versions use half=True
    print("quantize=16 did not work:", error)
    print("Trying half=True instead")
    path = model.export(format="engine", device=0, imgsz=imgsz, half=True)
os.replace(path, "mine.engine")
print(f"Built mine.engine in {time.time() - start:.0f} s, size {os.path.getsize('mine.engine') / 1e6:.1f} MB")

In [ ]:
with labview.build_lock():
    !python export_mine.py {MODEL_PT} {IMGSZ}

### Step 2: Measure the Speed-Up
The cell times the `.pt` model and the engine on the same picture: 5 warm-up runs, then 50 timed runs. It prints the time of one picture and the frames per second (FPS).

In [ ]:
from ultralytics import YOLO

picture = next(iter(glob.glob("../lab14/dataset/images/train/*.jpg")), None) or next(
    (p for p in ("../models/bus.jpg", "../lab8/bus.jpg") if os.path.exists(p)), None)
frame = cv2.imread(picture)
print("Test picture:", picture)


def benchmark(path, runs=50):
    model = YOLO(path, task="detect")
    for _ in range(5):
        model(frame, imgsz=IMGSZ, device=0, verbose=False)
    started = time.time()
    for _ in range(runs):
        model(frame, imgsz=IMGSZ, device=0, verbose=False)
    return 1000 * (time.time() - started) / runs


results = {}
for path in (MODEL_PT, "mine.engine"):
    if os.path.exists(path):
        results[path] = benchmark(path)
        print(f"{path:<14} {results[path]:6.1f} ms per picture   {1000 / results[path]:5.1f} FPS")
    else:
        print(path, "not found. Run Step 1 first.")
if len(results) == 2:
    print(f"The engine is {results[MODEL_PT] / results['mine.engine']:.1f} times faster.")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Time of the `.pt` model (ms and FPS):
- Time of the engine (ms and FPS):
- Speed-up:
- Is a speed-up important for this application? Why or why not?:


---
# Part C: Build the Alarm (50 min)

### Step 1: The Settings
`REQUIRED` are the items that must be on the board. For your own model, use the class names of Lab 14. `ZONE` is the board as a polygon of corners, as (x, y) fractions of the picture (like Lab 12). `HOLD` is the number of seconds an item must be absent (or back) before the state changes.

In [ ]:
# ---------------- Settings you can change ----------------
REQUIRED = ["mug", "toy"]     # the items that must be on the board (class names of your model)
ZONE = [(0.05, 0.10), (0.95, 0.10), (0.95, 0.95), (0.05, 0.95)]    # the board: corners as (x, y) fractions
HOLD = 3.0                    # seconds an item must be absent (or back) before the alarm or the all-clear
CONFIDENCE = 0.50             # confidence limit
MODEL = "mine.engine" if os.path.exists("mine.engine") else MODEL_PT
# ----------------------------------------------------------
print("Required:", REQUIRED, "| hold", HOLD, "s | model", MODEL)

In [ ]:
model = YOLO(MODEL, task="detect")
unknown = [name for name in REQUIRED if name not in model.names.values()]
print("Classes of the model:", list(model.names.values()) if len(model.names) < 12 else str(len(model.names)) + " classes")
if unknown:
    print("These required items are not a class of this model:", unknown, "-> check REQUIRED")
else:
    print("All required items are classes of the model.")

### Step 2: The Program
Read this cell before you run it.

| Function | What it does |
|---|---|
| `zone_polygon(shape)` | The `ZONE` in pixels for a picture of this size |
| `items_on_board(found, shape)` | From the detections (class name, box) it returns the set of required items whose box **centre** is inside the zone |
| `update_items(on_board, now)` | The hold rule for each required item. Returns a list of events `(item, "MISSING" or "BACK", seconds)` when a state changes. |
| `log_event(item, event, seconds)` | Adds a line to `board_log.csv` and rewrites `board_log.xlsx` |
| `process(frame, state)` | Detects, checks, raises the alarm and draws. The live view calls it for every picture. |

In [ ]:
items = {name: {"state": "OK", "since": time.time(), "cand": None, "cand_since": 0.0} for name in REQUIRED}
run = {"note": "", "warned": False, "client": None}


def zone_polygon(shape):
    h, w = shape[:2]
    return np.array([[int(x * w), int(y * h)] for x, y in ZONE], dtype=np.int32)


def items_on_board(found, shape):
    """found: a list of (class name, x1, y1, x2, y2). Returns the set of REQUIRED items whose box centre is inside the zone."""
    zone = zone_polygon(shape).reshape(-1, 1, 2)
    on_board = set()
    for name, x1, y1, x2, y2 in found:
        if name in REQUIRED and cv2.pointPolygonTest(zone, ((x1 + x2) / 2, (y1 + y2) / 2), False) >= 0:
            on_board.add(name)
    return on_board


def update_items(on_board, now):
    """The hold rule for each required item. Returns a list of (item, event, seconds in the old state)."""
    events = []
    for name, item in items.items():
        wanted = "OK" if name in on_board else "MISSING"
        if wanted == item["state"]:
            item["cand"] = None
        elif item["cand"] != wanted:
            item["cand"], item["cand_since"] = wanted, now
        elif now - item["cand_since"] >= HOLD:
            events.append((name, "MISSING" if wanted == "MISSING" else "BACK", now - item["since"]))
            item.update(state=wanted, since=now, cand=None)
    return events


def write_excel():
    """Rewrite board_log.xlsx from board_log.csv, so it opens in Excel with a header and number columns."""
    try:
        from openpyxl import Workbook
        from openpyxl.styles import Font
    except ImportError:
        if not run["warned"]:
            run["warned"] = True
            print("board_log.xlsx needs openpyxl. board_log.csv works in Excel too. Ask your instructor.")
        return
    book = Workbook()
    sheet = book.active
    sheet.title = "Board"
    with open("board_log.csv", newline="") as f:
        rows = list(csv.reader(f))
    for i, row in enumerate(rows):
        if i > 0:
            row[3] = float(row[3])
        sheet.append(row)
    for cell in sheet[1]:
        cell.font = Font(bold=True)
    for column, width in zip("ABCD", (20, 14, 12, 26)):
        sheet.column_dimensions[column].width = width
    book.save("board_log.xlsx")


def log_event(item, event, seconds):
    first = not os.path.exists("board_log.csv")
    with open("board_log.csv", "a", newline="") as f:
        writer = csv.writer(f)
        if first:
            writer.writerow(["time", "item", "event", "seconds_in_previous_state"])
        writer.writerow([datetime.now().strftime("%Y-%m-%d %H:%M:%S"), item, event, round(seconds, 1)])
    write_excel()


def send_alarm(item, event):
    """Optional: tell the phone. Works only after Part D Step 1 made run['client']."""
    if run["client"] is not None:
        try:
            run["client"].publish("board/alarm", f"{item} {event}")
        except Exception as error:
            print("Could not send to MQTT:", error)


def process(frame, state):
    result = model(frame, imgsz=IMGSZ, conf=state["Confidence"], device=0, verbose=False)[0]
    shown = frame.copy()
    found = []
    for box, cls in zip(result.boxes.xyxy.tolist(), result.boxes.cls):
        x1, y1, x2, y2 = (int(v) for v in box)
        name = result.names[int(cls)]
        found.append((name, x1, y1, x2, y2))
        cv2.rectangle(shown, (x1, y1), (x2, y2), (0, 255, 0) if name in REQUIRED else (160, 160, 160), 2)
        cv2.putText(shown, name, (x1, max(y1 - 6, 14)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)

    now = time.time()
    for item, event, seconds in update_items(items_on_board(found, frame.shape), now):
        log_event(item, event, seconds)
        send_alarm(item, event)
        run["note"] = f"{item} {event} at {datetime.now().strftime('%H:%M:%S')}"

    missing = [name for name, item in items.items() if item["state"] == "MISSING"]
    colour = (0, 0, 255) if missing else (0, 200, 0)
    cv2.polylines(shown, [zone_polygon(frame.shape)], True, colour, 3)
    cv2.putText(shown, "ALARM: " + ", ".join(missing) + " missing" if missing else "All items on the board", (10, 32),
                cv2.FONT_HERSHEY_SIMPLEX, 0.85, colour, 3)
    y = 62
    for name, item in items.items():
        text = f"{name}: {item['state']}"
        if item["cand"]:
            text += f"  (changing in {max(0.0, HOLD - (now - item['cand_since'])):.1f} s)"
        cv2.putText(shown, text, (10, y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255) if item["state"] == "MISSING" else (255, 255, 255), 2)
        y += 24
    cv2.putText(shown, f"{state['fps']:.0f} FPS", (shown.shape[1] - 110, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
    if run["note"]:
        cv2.putText(shown, run["note"], (10, shown.shape[0] - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)
    return shown


print("The program is ready.")

### Step 3: Check the Logic
Before the live test, check `items_on_board` and `update_items` with fake detections. Item `A` is inside the zone and item `B` is outside, so only `A` is on the board.

In [ ]:
saved_required, saved_items = list(REQUIRED), dict(items)
REQUIRED[:] = ["A", "B"]
items.clear()
items.update({n: {"state": "OK", "since": 0.0, "cand": None, "cand_since": 0.0} for n in REQUIRED})

shape = (480, 640, 3)
found = [("A", 300, 200, 340, 240), ("B", 0, 0, 20, 20)]            # B is in the corner, outside the zone
print("On the board:", items_on_board(found, shape), "(expected {'A'})")
print(update_items({"A"}, 10.0), "<- B starts to be missing: no event yet")
print(update_items({"A"}, 11.0), "<- still waiting")
print(update_items({"A"}, 13.5), "<- after HOLD seconds: B is MISSING")
print(update_items({"A", "B"}, 14.0), update_items({"A", "B"}, 17.5), "<- B is back after HOLD seconds")

REQUIRED[:] = saved_required                                         # put the real settings back
items.clear()
items.update(saved_items)

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Did all the lines give the expected result?:
- How long was B missing when the event was reported?:
- Why does the program need a separate state for each item?:


### Step 4: Run the Alarm
Run the cell. Put the required items on the board (inside the green outline). Take one away for longer than `HOLD` seconds: the outline turns red and `ALARM` shows with the item name. Put it back: after `HOLD` seconds the alarm clears. Take an item away for only one second: nothing should happen. Press **Stop** when you are done.

In [ ]:
now = time.time()
for item in items.values():
    item.update(state="OK", since=now, cand=None)
run["note"] = ""
live = labview.live(process, controls={"Confidence": (0.05, 0.95, 0.05, CONFIDENCE)})

### Step 5: See the Results
Run this cell after you pressed Stop. To open the Excel file, use the file browser on the left, right-click `board_log.xlsx`, choose **Download** and open it on your laptop.

In [ ]:
def show_results():
    for name in ("board_log.csv", "board_log.xlsx"):
        print(f"{name:<16}", "found" if os.path.exists(name) else "not created yet (it appears at the first event)")
    if os.path.exists("board_log.csv"):
        print()
        with open("board_log.csv") as f:
            print(f.read())


show_results()

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- FPS shown on the screen (compare with Part B):
- How many seconds did it take from taking an item away to the alarm? Does it match HOLD?:
- What happened when you took an item away for one second?:
- First line of the log after the header:


---
# Part D: Send the Alarm to Your Phone and Make It Your Own (35 min)

### Step 1: Send to the Phone with MQTT (optional)
Your Jetson has a personal MQTT broker. Open **Details** on your student card on the teacher page to see the host, the port, the user and the password. In the app *IoT MQTT Panel* on your phone, connect to that broker and add a **Text** (log) panel that subscribes to the topic `board/alarm`. Then run the cell. After that, every missing and returned event is also sent to the phone.

In [ ]:
try:
    run["client"] = labview.mqtt()
    print("Connected. Messages go to the topic board/alarm.")
    print("Broker details:", {k: v for k, v in labview.mqtt_info().items() if k != "password"})
    run["client"].publish("board/alarm", "board alarm ready")
except SystemExit as error:
    print(error)
except Exception as error:
    print("No MQTT connection:", error)

Run Step 4 of Part C again and take an item away. Look at the phone.

### Step 2: Make It Your Own
Add **at least one improvement** from the list, or one of your own. Change the code in the cells above, run that cell again, then run the live cell again.

- Add or remove items in `REQUIRED`, or change the `ZONE` to the shape of your board.
- Show how long an item has been missing in minutes and seconds on the screen.
- Make a **count of alarms** per item and show it on the screen.
- Raise an alarm only when **two or more** items are missing, or when a **special** item is missing at once.
- Send a second message when everything is back on the board.
- Receive a message **from** the phone, for example `reset`, to clear the alarm. See `labview.mqtt(on_message, subscribe)` in Lab 9's notes.

> **Tip.** Change one thing at a time and run it after each change. To keep a copy of a cell before a big change, copy the cell (select it, press **C**, then **V**) and change the copy.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Did the phone get the messages? What did it show?:
- Improvement you chose:
- Which lines did you change?:
- Did it work the first time? What went wrong?:


---
# Part E: Test and Present (30 min)

### Step 1: Test the Alarm
Delete the old log, run the alarm again and test each situation.

In [ ]:
for name in ("board_log.csv", "board_log.xlsx"):
    if os.path.exists(name):
        os.remove(name)
print("Clean. Files left:", sorted(glob.glob("board_log*")))

| Test | Situation | Expected | What happened |
|---|---|---|---|
| 1 | All items on the board for 10 seconds | No alarm | |
| 2 | One item taken away for 1 second | No alarm (shorter than `HOLD`) | |
| 3 | One item taken away for 10 seconds | ALARM after about `HOLD` seconds, one log line | |
| 4 | The item put back | Alarm clears after about `HOLD` seconds, one log line | |
| 5 | Two items taken away | ALARM with both names | |
| 6 | An item put **beside** the board (outside the zone) | ALARM (not on the board) | |
| 7 | An item that is not in `REQUIRED` put on the board | No change | |
| 8 | Dim light | Compare with test 1 | |

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Test 1 to 4: were all four as expected?:
- Test 5 and 6:
- Test 7 and 8:
- HOLD and CONFIDENCE that worked best:


### Step 2: Report and Present
Fill in the report. Then give a **3-minute demonstration**: show the speed of the `.pt` model and the engine, the live board with an alarm and the all-clear, the log in Excel and the phone message.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Board idea and required items:
- Speed: `.pt` model and engine (ms and FPS) and the speed-up:
- Power mode used (run  !sudo nvpmodel -q  in a cell, after  labview.sudo_login()  ):
- Your improvement:
- One problem you found and how you fixed it:
- One idea to make it better in future:


> **Where next.** This was the last of the vision labs. Next you give the Jetson ears: **speech to text**, **keyword spotting** and **text to speech**.

## 4. Check Task
Show your instructor. The instructor initials each line when it is done correctly.

| No. | Task to demonstrate | Initials |
|---|---|---|
| 1 | `mine.engine` is built and the speed of the `.pt` model and the engine is compared | |
| 2 | The logic check of Part C Step 3 gives the expected results | |
| 3 | The live view shows the board outline and the state of each item | |
| 4 | Taking an item away for longer than `HOLD` raises the alarm, and a short absence does not | |
| 5 | Putting the item back clears the alarm and logs how long it was missing | |
| 6 | The log is shown in Excel (`board_log.xlsx`) with at least 2 lines | |
| 7 | At least one improvement works and you explain the code you changed | |
| 8 | The test table is complete | |
| 9 | The report is complete and the 3-minute presentation is given | |

## 5. Assessment Questions
Answer in one or two sentences.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- 1. Why is an engine built for one picture size and for one kind of Jetson?:
- 2. What did you gain from the engine? Give the numbers from Part B.:
- 3. How does the program decide that an item is on the board?:
- 4. Why does each item have its own hold time and state?:
- 5. The alarm rings every time somebody lifts a tool for a second. Which setting do you change, and what is the cost?:
- 6. Why does the program log how long the item was missing?:
- 7. Give two reasons why a model could miss an item that is on the board.:
- 8. What is MQTT used for in this lab, and what is a topic?:
- 9. Which is the best use of this lab's idea? (a) Checking that all the tools are back on the board at the end of the day (b) Reading a number plate (c) Measuring a temperature (d) Playing music:


## 6. Troubleshooting

| Symptom | Likely cause | What to try |
|---|---|---|
| `mine.engine not found` in Part B | The build was not run, or it failed | Run Step 1 of Part B again and wait for `Built mine.engine` |
| The build says `waiting` | Another student builds or trains | Wait. Do not stop the cell. |
| The engine gives wrong or no detections | The picture size does not match the model | Use the same `IMGSZ` for the build and for `model(...)` (320 for your own model) |
| `CUDA out of memory` | Other notebooks use the shared 8 GB | Press Stop in all live views, Kernel, Shut Down All Kernels, then run the notebook again from the top |
| `These required items are not a class of this model` | A name in `REQUIRED` is spelled differently | Use the exact class names printed above (the model of Lab 14 uses your own names) |
| The alarm is always on | The item is not detected, is outside the zone, or `CONFIDENCE` is too high | Lower the confidence slider. Check that the item is inside the green outline. |
| The alarm never comes | `HOLD` is too long, or the item is still detected after it is gone | Lower `HOLD`. Check what the green boxes show. |
| The status flickers | `HOLD` is too short or the detection is unstable | Raise `HOLD`. Improve the model with more pictures (Lab 14). |
| No message on the phone | The phone is not connected to the broker, or the topic is different | Check the details on the teacher page and the topic `board/alarm` in the app |
| `Cannot open the camera` | Another program uses the camera | Press Stop in other live cells, shut down other kernels, ask your teacher to check your camera on the teacher page |
| The notebook looks broken | You changed or deleted too much | Close it and run `import labview; labview.reset(15)` in a cell of a new notebook (File, New, Notebook). You get a fresh copy. |

## 7. Further Reading
- Ultralytics, Export mode: https://docs.ultralytics.com/modes/export/
- NVIDIA TensorRT: https://developer.nvidia.com/tensorrt
- OpenCV, pointPolygonTest: https://docs.opencv.org/4.x/d3/dc0/group__imgproc__shape.html
- MQTT basics: https://mqtt.org/